In [2]:
import rdflib as rdf
from rdflib import Graph
import torch
from collections import Counter

In [3]:
g = Graph()
g.parse("dataset/aifbfixed_complete.n3")

print("Graph loaded successfully!")

Graph loaded successfully!


# Node - Create Node IDs

Convert --  (person1, publication, paper1) ----> (0, publication, 1)

In [5]:
from rdflib import Literal

all_nodes = set()

for s,p,o in g:

    # subject is a node
    all_nodes.add(s)
    # Only add Object if not literal
    if not isinstance(o, Literal):
        all_nodes.add(o)

print("Total unique nodes", len(all_nodes))




# Create Node-to-ID Mapping
node_to_id = {}

for idx, node in enumerate(all_nodes):
    node_to_id[node] = idx




# Create Reverse Mapping
id_to_node = {}

for node, idx in node_to_id.items():
    id_to_node[idx] = node



# Test the Mapping
print("\nSample Node IDs:\n")
for i, (node, idx) in enumerate(node_to_id.items()):
    print(idx, node)

    if i >= 10:
        break

Total unique nodes 2835

Sample Node IDs:

0 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id1273instance
1 http://www.aifb.uni-karlsruhe.de/Publikationen/viewExternerAutorOWL/id824instance
2 http://www.aifb.uni-karlsruhe.de/Forschungsgebiete/viewForschungsgebietOWL/id40instance
3 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id18instance
4 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id577instance
5 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id233instance
6 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id1003instance
7 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id319instance
8 http://www.aifb.uni-karlsruhe.de/Personen/viewPersonOWL/id29instance
9 http://www.aifb.uni-karlsruhe.de/Publikationen/viewExternerAutorOWL/id361instance
10 http://www.aifb.uni-karlsruhe.de/Publikationen/viewPublikationOWL/id469instance


# Edge - Create Predicate IDs

In [6]:
predicate_to_id = {}

relation_id = 0

for s, p, o in g:

    # Skip Literals
    if isinstance(o, Literal):
        continue

    if p not in predicate_to_id:
        predicate_to_id[p] = relation_id
        relation_id += 1

print("Total Relation Types :- ", len(predicate_to_id))

Total Relation Types :-  24


# Edge - RDF Triple into Edge IDs & Edge Types

In [7]:
# RDF triple into Edge Ids

REMOVE_RELATIONS = [
    "http://swrc.ontoware.org/ontology#affiliation",
    "http://swrc.ontoware.org/ontology#employs"
]

edges = []
edge_types = []

for s, p, o in g:

    # Skip Literals
    if isinstance(o, Literal):
        continue

    # Remove Leakage Relations
    if(str(p) in REMOVE_RELATIONS):
        continue

    source = node_to_id[s]
    target = node_to_id[o]

    relation_id = predicate_to_id[p]

    edges.append((source, target))
    edge_types.append(relation_id)

print(edges[0:10])
print(edge_types) # FIND PURPOSE OF IT ???????.....




[(987, 2737), (1279, 2599), (1015, 187), (1996, 531), (1339, 1910), (1631, 730), (1163, 2819), (2480, 1661), (8, 553), (810, 290)]
[0, 1, 2, 2, 3, 1, 4, 5, 6, 7, 4, 8, 8, 7, 2, 2, 2, 4, 1, 4, 7, 4, 1, 4, 4, 9, 6, 7, 3, 7, 10, 9, 0, 11, 4, 2, 2, 7, 1, 1, 4, 4, 0, 1, 4, 13, 6, 7, 0, 1, 14, 2, 7, 5, 4, 2, 1, 2, 1, 2, 5, 7, 4, 4, 2, 1, 1, 1, 5, 10, 1, 7, 2, 4, 4, 4, 5, 1, 4, 9, 4, 2, 5, 1, 4, 2, 7, 0, 15, 4, 2, 0, 1, 9, 5, 4, 1, 1, 0, 2, 1, 1, 1, 4, 2, 2, 8, 2, 5, 4, 7, 2, 0, 1, 1, 7, 7, 0, 7, 16, 7, 10, 9, 1, 1, 2, 1, 1, 17, 1, 4, 2, 7, 5, 2, 4, 2, 7, 4, 1, 10, 10, 10, 1, 2, 4, 14, 2, 10, 18, 1, 7, 4, 4, 4, 8, 2, 1, 1, 4, 4, 7, 13, 1, 4, 1, 4, 8, 2, 2, 4, 5, 1, 7, 2, 1, 8, 7, 1, 5, 10, 19, 4, 8, 1, 8, 0, 1, 1, 18, 7, 4, 15, 4, 4, 2, 7, 6, 4, 0, 4, 4, 4, 9, 6, 1, 0, 4, 5, 10, 5, 2, 16, 9, 4, 7, 4, 4, 16, 2, 7, 2, 3, 1, 2, 5, 8, 4, 0, 7, 1, 4, 5, 2, 4, 7, 9, 7, 1, 7, 5, 8, 4, 7, 1, 4, 1, 7, 2, 8, 4, 1, 4, 7, 5, 3, 7, 2, 1, 7, 5, 1, 8, 4, 1, 4, 6, 5, 1, 1, 4, 2, 2, 1, 2, 7, 4, 7, 4, 4, 15, 0

In [8]:
# Seperate Edges 

edge_ids = []
for s, p, o in g:
    if isinstance(o, Literal):
        continue
    source = node_to_id[s]
    target = node_to_id[o]
    edge_id = len(edge_ids)
    edge_ids.append((edge_id, source, target, p))

print(edge_ids[0:10])

[(0, 987, 2737, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#hasProject')), (1, 1279, 2599, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#publication')), (2, 1015, 187, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#author')), (3, 1996, 531, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#author')), (4, 1339, 1910, rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf')), (5, 1631, 730, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#publication')), (6, 1163, 2819, rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type')), (7, 2480, 1661, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#publishes')), (8, 8, 553, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#worksAtProject')), (9, 810, 290, rdflib.term.URIRef('http://swrc.ontoware.org/ontology#isAbout'))]


# Edge - Create Edge_index & Edge Type Tensors

PyTorch Geometric required format - [ 2, num_edges ]

What is edge_index ?

PyTorch Geometric stores graph edges like this :--
    [
 [source_nodes],
 [target_nodes]
]

NOT as tuples.

Python List ----> PyTorch Tensor

In [9]:
# Edge Index
edge_index = torch.tensor(
    edges,
    dtype = torch.long
).t().contiguous()

print("Edge Index Shape : ", edge_index.shape)
print("edge_index : ",edge_index[ :, :10 ])


# Convert Edge Type into Tensors
edge_type = torch.tensor(
    edge_types,
    dtype = torch.long
)

print("Edge Type Shape : ", edge_type.shape)
print("edge_type : ", edge_type)


Edge Index Shape :  torch.Size([2, 20338])
edge_index :  tensor([[ 987, 1279, 1015, 1996, 1339, 1631, 1163, 2480,    8,  810],
        [2737, 2599,  187,  531, 1910,  730, 2819, 1661,  553,  290]])
Edge Type Shape :  torch.Size([20338])
edge_type :  tensor([0, 1, 2,  ..., 4, 4, 4])


# Label - Define Label Predicate 

Model Predict Research Group

Hence Research Group affiliation used

In [10]:
LABEL_PREDICATE = "http://swrc.ontoware.org/ontology#affiliation"

# Collect unique Labels
unique_labels = set()

for s, p, o in g:
    if str(p) == LABEL_PREDICATE:
        unique_labels.add(o)

print("Classes :- ", unique_labels)
print("Total Classes :- ", len(unique_labels))



# Create label IDs
label_to_id = {

    label : idx
    for idx, label in enumerate(unique_labels)
}
print("Label IDs :- ",label_to_id)



Classes :-  {rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id2instance'), rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id4instance'), rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id5instance'), rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id1instance'), rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id3instance')}
Total Classes :-  5
Label IDs :-  {rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id2instance'): 0, rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id4instance'): 1, rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungsgruppen/viewForschungsgruppeOWL/id5instance'): 2, rdflib.term.URIRef('http://www.aifb.uni-karlsruhe.de/Forschungs

# Label - Create Node Labels

In [11]:
labels = {}

for s, p, o in g:

    if str(p) == LABEL_PREDICATE:

        node_id = node_to_id[s]
        class_id = label_to_id[o]
        labels[node_id] = class_id

print(node_id)
print(class_id)
print(labels)

976
4
{2465: 3, 1203: 4, 1769: 3, 2468: 3, 1509: 3, 342: 4, 728: 3, 2569: 3, 809: 3, 767: 1, 708: 4, 1044: 4, 2621: 4, 1229: 0, 2016: 4, 1060: 0, 1009: 4, 1534: 4, 2358: 3, 1917: 0, 96: 4, 727: 3, 184: 3, 705: 4, 2140: 4, 1138: 4, 2342: 3, 2119: 4, 2292: 4, 1552: 3, 1267: 4, 2590: 1, 334: 1, 1770: 4, 1733: 3, 300: 4, 2131: 3, 1717: 4, 2456: 3, 366: 3, 1442: 1, 2355: 3, 1182: 4, 1407: 0, 1021: 4, 1779: 3, 1750: 4, 1260: 3, 103: 3, 8: 0, 2773: 1, 1134: 4, 2769: 3, 343: 3, 192: 1, 1693: 0, 1950: 0, 2099: 3, 2434: 3, 2637: 3, 2431: 1, 2263: 3, 1683: 3, 2333: 3, 2078: 3, 1936: 4, 815: 1, 2594: 0, 2320: 3, 11: 4, 401: 0, 1866: 4, 2183: 3, 63: 3, 799: 4, 1175: 4, 654: 3, 2495: 3, 2408: 3, 2780: 3, 1809: 3, 533: 1, 564: 4, 268: 0, 1173: 0, 1279: 3, 253: 0, 842: 3, 453: 4, 222: 3, 2307: 3, 808: 1, 2393: 4, 1188: 1, 1166: 0, 1399: 4, 1037: 4, 879: 0, 678: 3, 1913: 4, 340: 0, 356: 0, 2576: 4, 275: 3, 1764: 2, 1658: 1, 2489: 3, 1839: 3, 2349: 4, 840: 4, 1072: 4, 2725: 1, 502: 0, 711: 3, 1066: 0, 2

# Label - Convert Labels to Tensors

In [12]:
num_nodes = len(node_to_id)

y = torch.full(
    (num_nodes,),
    -1,
    dtype = torch.long
)

for node_id, class_id in labels.items():
    y[node_id] = class_id

print(y.shape);
print(y[:20]);

torch.Size([2835])
tensor([-1, -1, -1, -1, -1, -1, -1, -1,  0, -1, -1,  4, -1, -1, -1, -1, -1, -1,
        -1, -1])


In [13]:
print(torch.unique(y))

tensor([-1,  0,  1,  2,  3,  4])
